In [48]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3 as sql

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [ ]:
DB = "prices.db"
with sql.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
        'CREATE TABLE IF NOT EXISTS prices ('
        'id INTEGER PRIMARY KEY AUTOINCREMENT,'
        'destination TEXT NOT NULL,'
        'price REAL NOT NULL'
        ')'
    )
    conn.commit()


def set_ticket_price(destination, price):
    with sql.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT OR REPLACE INTO prices (destination, price) VALUES (?, ?)',
            (destination.lower(), price)
        )
        conn.commit()
set_ticket_price("london", "$500")
set_ticket_price("paris", "$450")
set_ticket_price("new york", "$600")
set_ticket_price("tokyo", "$700")
set_ticket_price("sydney", "$800")

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.

Only use get_ticket_price when the user asks for a ticket price.
For greetings, respond naturally without using any tool.
Do not make up ticket prices.
Give short, courteous answers, no more than 1 sentence.
"""

def get_ticket_price(destination):
    print(f"Database Tool called for city {destination}")
    with sql.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE destination = ?', (destination.lower(),))
        row = cursor.fetchone()
        if row:
            return f'The price for a ticket to {destination} is {row[0]}'
        return f'Sorry, ticket price for {destination} is not available'

In [44]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

In [45]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            destination_city = arguments.get("destination_city")
            price_details = get_ticket_price(destination_city)

            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses

In [46]:
def chat(message, history):

    history = [
        {"role": h["role"], "content": h["content"]}
        for h in history
    ]

    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [47]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Tool called for city Paris
Tool called for city New York
Tool called for city london
Tool called for city paris
